In [0]:
employee_df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv", header=True, inferSchema=True, sep="|", quote="'")
employee_df.display()

#Selecting Columns

##using string

In [0]:
employee_df.select("id", "name").display()

##using column object

In [0]:
#using column objects
from pyspark.sql.functions import col
employee_df.select(col("id"), col("name")).display()

###change the column name

In [0]:
from pyspark.sql.functions import col
employee_df.select(col("id"), col("name").alias("full_name")).display()

In [0]:
employee_df.columns

##Exclude columns

###Imperative Style

In [0]:
columns_to_exclude=["dob"]
columns_to_include=[]
for column in employee_df.columns:
    if column not in columns_to_exclude:
        columns_to_include.append(column)

employee_df.select(*columns_to_include).display()

###Functional Style

even_numbers=[1, 2, 3, 4, 5, 6, 7, 8]
for e in L:
    if e%2==0:
        even_numbers.append(e)

even_numbers
[2, 4, 6, 8]

list(filter(lambda e:e%2==0, even_numbers))

even_numbers
[2, 4, 6, 8]

In [0]:
columns_to_exclude=["dob"]
columns_to_include=list(filter(lambda column: column not in columns_to_exclude, employee_df.columns))

employee_df.select(*columns_to_include).display()

###Composition style (recommended in python)

In [0]:
# [e for e in L if e%2==0]
columns_to_exclude=["dob"]
columns_to_include=[column for column in employee_df.columns if column not in columns_to_exclude]

employee_df.select(*columns_to_include).display()

#Filter operations

In [0]:
#display(employee_df.filter(col("gen")=="M"))
employee_df.filter(col("gen")=="M").select("name").display()
#first select gender column with M and then select name column for M genders

In [0]:
#Filter those records whose gender is M and whose experience is greater than 2
employee_df.filter(col("gen")=="M").filter(col("exp")>2).select("id").display()

In [0]:
#employee name with complany cisco
employee_df.filter(col("company")=="cisco").select("name").display() #currently getting 1 but have 2 in table due to case sensitive


In [0]:
from pyspark.sql.functions import lower
employee_df.filter(lower("company")=="cisco").select("name").display()

In [0]:
#employee_df.filter((col("desig")=="Team Lead") | (col("desig")=="Developer")).display()

employee_df.filter(col("desig").isin(["Team Lead", "Developer"])).display()

# Grouping, Aggregations and Sorting

In [0]:
#employee_df.groupBy("gen").count().sort(col("count").desc()).display()
employee_df.groupBy("gen").count().orderBy(col("count"), ascending=False).display()

#Derive new columns

##using .withColumn

In [0]:
from pyspark.sql.functions import lit
employee_df.withColumn("is_employed", lit(True)).display()

##Using .withColumn and Conditionas

In [0]:
from pyspark.sql.functions import when
employee_df.withColumn("exp_level", when(col("exp")>=10, "Senior").when(col("exp")>=5, "Junior").when(col("exp")>=0, "Junior").otherwise("Invalid Experience")).select("name", "exp", "exp_level").display()

##Using. selectExpr

In [0]:
employee_df.selectExpr(
    "*",
    """ 
    CASE 
    WHEN exp>=10 THEN 'Senior' 
    WHEN exp>=5 THEN 'Mid-Level' 
    WHEN exp>=0 THEN 'Junior' 
    ELSE 'Invalid Experience' 
    END AS exp_level""",
).display()

#Create a view

In [0]:
employee_df.createOrReplaceTempView("employee_vw")

In [0]:
%sql
select name, exp, CASE 
    WHEN exp>=10 THEN 'Senior' 
    WHEN exp>=5 THEN 'Mid-Level' 
    WHEN exp>=0 THEN 'Junior' 
    ELSE 'Invalid Experience' 
    END AS exp_level
    from employee_vw;